# Лабораторная работа 3. Градиентный спуск своими руками

**Курс «Машинное обучение», 4 курс** · после лекции 2

## Зачем это нужно

Обучить модель — значит подобрать параметры $\theta$, при которых ошибка $Q(\theta)$ минимальна. Для линейной регрессии это можно сделать формулой (нормальные уравнения, прошлое занятие), но:

- для больших данных обращать матрицу $X^\top X$ слишком дорого;
- для большинства моделей (логистическая регрессия, нейросети, бустинг) формулы вообще нет.

Поэтому почти всё машинное обучение держится на одном приёме: **градиентном спуске**.

У градиентного спуска есть главная настройка — **длина шага $\eta$**. Слишком маленький — учимся вечно, слишком большой — обучение «взрывается».

## Одна идея на всё занятие

> **Градиентный спуск — это спуск с горы в тумане: мы видим только наклон под ногами и делаем шаг вниз.**
> Всё занятие отвечает на один вопрос: **какой шаг $\eta$ можно делать, чтобы дойти до дна, а не улететь?**

**Данные:** на занятии — **сгенерированные «квартиры»**: цена (млн ₽) по площади, числу комнат, этажу и расстоянию до метро. Мы сами задаём «истинные» веса, поэтому в конце сможем проверить, нашла ли их модель.

**План:**

| Часть | Вопрос |
|---|---|
| 0 | Данные «квартиры»: что хотим предсказать; вспоминаем прошлое занятие |
| 1 | Как шаг влияет на спуск (одна переменная) |
| 2 | То же в «вытянутой долине» (две переменные) |
| 3 | Проверяем порог шага на регрессии |
| 4 | Почему надо стандартизовать признаки |
| 5 | **Ваше задание:** пишем спуск сами |
| 6 | SGD: когда он выигрывает и как его применять |
| 7 | Регуляризация: Ridge и LASSO |
| 8 | Найди ошибку: шаг «перенесли» не туда |

**Как работать:** перед каждой частью — вопрос **«Подумайте»**. Сначала ответьте (вслух или в тетради), потом запускайте ячейки и сравнивайте. Все ячейки уже написаны, кроме отмеченных `✍` (проба шагов в части 1 и задание в части 5).

In [ ]:
# Запустите эту ячейку и не читайте её: здесь только импорты и настройки графиков.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.grid'] = True          # сетка на всех графиках

In [ ]:
def make_flats(n, seed=0):
    # генератор «квартир»: цена = 1 + 0.10·площадь + 0.30·комнаты + 0.02·этаж − 0.40·метро + шум
    rng = np.random.default_rng(seed)
    area = np.clip(rng.normal(70, 25, n), 25, 160)
    rooms = np.clip(np.round(area / 28 + rng.normal(0, 0.5, n)), 1, 6)
    floor = rng.integers(1, 26, n).astype(float)
    metro_km = np.clip(rng.gamma(2.0, 1.3, n), 0.1, 12)
    price = 1.0 + 0.10 * area + 0.30 * rooms + 0.02 * floor - 0.40 * metro_km + rng.normal(0, 1.0, n)
    return pd.DataFrame({'area': area.round(1), 'rooms': rooms, 'floor': floor,
                         'metro_km': metro_km.round(2), 'price': price.round(2)})


TRUE_WEIGHTS = {'area': 0.10, 'rooms': 0.30, 'floor': 0.02, 'metro_km': -0.40}   # «истина», по которой построена цена

---
# Часть 0. Данные и прошлое занятие

На прошлом занятии мы решали линейную регрессию **формулой**: нормальные уравнения $\theta^*=(X^\top X)^{-1}X^\top y$. Функция потерь там — $Q(\theta)=\|X\theta-y\|^2=n\cdot\text{MSE}$; мы рисовали её линии уровня (эллипсы), а решение — это **дно «чаши»**. Данные брали со столбцом единиц для свободного члена.

Сегодня найдём то же самое дно **другим способом**: не формулой, а **шагами вниз**.

Сначала посмотрим на данные.

In [ ]:
flats = make_flats(300)                  # 300 квартир
features = ['area', 'rooms', 'floor', 'metro_km']
print(flats.head(4).to_string())

**Что мы хотим сделать с этими данными.** Задача — **предсказывать цену квартиры** по четырём признакам линейной моделью
$$\text{цена}\approx\theta_0+\theta_1\cdot\text{площадь}+\theta_2\cdot\text{комнаты}+\theta_3\cdot\text{этаж}+\theta_4\cdot\text{метро}.$$
Веса $\theta$ найдём градиентным спуском (в части 5 — своим кодом). Данные сгенерированы, поэтому в конце части 5 сверим найденные веса с истинными и сделаем прогноз для новой квартиры. Сначала посмотрим, как цена связана с каждым признаком:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.3), sharey=True)
names = {'area': 'площадь, м²', 'rooms': 'комнат', 'floor': 'этаж', 'metro_km': 'до метро, км'}
for ax, feat in zip(axes, features):
    ax.scatter(flats[feat], flats['price'], s=8, alpha=0.5)
    r = np.corrcoef(flats[feat], flats['price'])[0, 1]
    ax.set_xlabel(names[feat]); ax.set_title(f'корреляция с ценой: {r:.2f}', fontsize=9)
axes[0].set_ylabel('цена, млн ₽')
plt.tight_layout()

**Что видим:** цена почти линейно растёт с площадью — это главный признак; число комнат растёт вместе с площадью (признаки связаны между собой, это скажется в части 5); чем дальше метро, тем дешевле (связь слабее: её заслоняет разброс площадей); этаж влияет слабо.

Начнём с одного признака — **площади** — и решим задачу формулой, как на прошлом занятии:

In [ ]:
x = flats['area'].values
y = flats['price'].values
X1 = np.column_stack([np.ones(len(x)), x])                      # столбец единиц + площадь, как на прошлом занятии
theta_normal = np.linalg.solve(X1.T @ X1, X1.T @ y)              # нормальные уравнения (как ваша ols_fit)
print('\nрешение по формуле: θ = (сдвиг, наклон) =', theta_normal.round(3))

---
# Часть 1. Спуск в одной переменной

Разберёмся, как спускаться, на самом простом «рельефе»: параболе $f(x)=x^2$. Её дно в точке $x=0$, производная $f'(x)=2x$. Начнём **слева от дна**, с точки $x_0=-4$.

**Подумайте.** В какую сторону надо идти, чтобы функция уменьшалась? Куда покажет градиент (производная) в точке $-4$?

In [ ]:
def f(x):
    return x ** 2

def grad_f(x):
    return 2 * x

x0 = -4
print('значение f в точке -4:', f(x0))
print('градиент в точке -4: ', grad_f(x0))

Градиент равен $-8$: он **отрицательный**. **Градиент показывает, куда сдвинуть $x$, чтобы функция росла быстрее всего.** Поэтому он живёт **на оси $x$** (там, где мы выбираем точку), а не на графике и не по касательной. С касательной он связан так: её наклон в точке старта равен $f'(-4)=-8$ — это и есть градиент. Нарисуем его стрелкой на оси $x$ (пунктир — касательная):

In [ ]:
eta = 0.1
xs_curve = np.linspace(-13, 13, 200)

def plot_arrows(arrows, mark=False, tangent=False):
    # парабола, старт и дно; стрелки-векторы (сдвиг по x, цвет, подпись) рисуем НА ОСИ x — там живёт градиент
    plt.figure(figsize=(8, 4.5))
    plt.plot(xs_curve, f(xs_curve), 'r', label='f(x) = x²')
    plt.axvline(0, color='gray', ls='--', label='дно (x = 0)')
    plt.axhline(0, color='k', lw=0.8)
    plt.plot(x0, f(x0), '*k', ms=14, label='старт x0 = -4')
    plt.plot([x0, x0], [0, f(x0)], ':k', lw=1)                           # от старта вниз, на ось x
    if tangent:                                                           # касательная: её наклон и есть f'(x0)
        xt = np.array([x0 - 2.5, x0 + 2.5])
        plt.plot(xt, f(x0) + grad_f(x0) * (xt - x0), 'k--', lw=1, label=f'касательная, наклон {grad_f(x0)}')
    for dx, color, label in arrows:
        plt.annotate('', xy=(x0 + dx, 0), xytext=(x0, 0), arrowprops=dict(arrowstyle='->', color=color, lw=3))
        plt.plot([], [], color=color, lw=3, label=label)
        if mark:                                                          # куда попадаем: точка на оси и на параболе
            plt.plot([x0 + dx] * 2, [0, f(x0 + dx)], ':', color=color, lw=1)
            plt.plot(x0 + dx, f(x0 + dx), 'o', color=color, ms=11)
    plt.legend(fontsize=8); plt.xlabel('x'); plt.ylabel('f(x)')

plot_arrows([(grad_f(x0), 'tab:orange', 'градиент (−8): туда функция растёт')], tangent=True)

Стрелка смотрит влево: там функция растёт. Чтобы спуститься, нужно идти **против** градиента, то есть по **антиградиенту** (вправо). Добавим его на картинку:

In [ ]:
plot_arrows([(grad_f(x0), 'tab:orange', 'градиент (−8)'),
             (-grad_f(x0), 'tab:blue', 'антиградиент (+8): туда идём')])

Шагнуть на весь антиградиент нельзя: окажемся в $-4+8=4$, на той же высоте по другую сторону от дна (перелёт). Поэтому антиградиент умножают на число $\eta$ (**длина шага**, learning rate). Для $\eta=0.1$ шаг в 10 раз короче: стрелка всего $+0.8$, и мы попадаем в точку $x_1=-3.2$ — ближе к дну, без перелёта. Приблизим картинку возле старта:

In [ ]:
plot_arrows([(-eta * grad_f(x0), 'tab:blue', f'η · антиградиент = +{-eta * grad_f(x0):.1f}')], mark=True)
plt.xlim(-5, 1); plt.ylim(-2, 30)
plt.title(f'Шаг η = {eta}: новая точка x = {x0 - eta * grad_f(x0):.1f} (синяя точка), ближе к дну');

Повторим шаг много раз по формуле
$$x_{t+1}=x_t-\eta\,f'(x_t).$$
Первые 5 шагов выведем в таблицу, чтобы увидеть числа.

In [ ]:
eta = 0.1
x_t = -4.0
path = [x_t]
print(' шаг |   x_t    | градиент |  x_(t+1)')
for t in range(10):
    g = grad_f(x_t)
    x_new = x_t - eta * g
    if t < 5:
        print(f'{t:4d} | {x_t:8.4f} | {g:8.4f} | {x_new:8.4f}')
    x_t = x_new
    path.append(x_t)
print('  ...')
print('после 10 шагов x =', round(x_t, 4))

Каждый следующий шаг короче предыдущего: чем ближе к дну, тем меньше градиент, а значит, и шаг.

### Когда остановиться?
Можно делать фиксированное число шагов. А можно остановиться, когда точка почти перестала двигаться: $|x_{t+1}-x_t|\le\varepsilon$. Это **критерий остановки**.

In [ ]:
eps = 1e-4
x_t = -4.0
steps = 0
while True:
    x_new = x_t - 0.1 * grad_f(x_t)
    steps += 1
    if abs(x_new - x_t) <= eps:
        break
    x_t = x_new
print(f'шагов до остановки: {steps}, x = {x_new:.6f}')

### Что будет, если шаг слишком большой?
Возьмём $\eta=1.1$ — чуть больше границы, которую мы вот-вот найдём. Формула шага та же, посмотрим на числа.

**Подумайте.** Приблизимся ли мы к дну? Что будет со значением $f$?

In [ ]:
eta_big = 1.1
x_t = -4.0
path_big = [x_t]
print(' шаг |   x_t    |  f(x_t)  | градиент |  x_(t+1)')
for t in range(6):
    g = grad_f(x_t)
    x_new = x_t - eta_big * g
    print(f'{t:4d} | {x_t:8.3f} | {f(x_t):8.2f} | {g:8.3f} | {x_new:8.3f}')
    x_t = x_new
    path_big.append(x_t)

**Что видим:** каждый шаг **перелетает** через дно и приземляется на стенку параболы, причём **выше**, чем стартовал: $f$ было 16, потом 23, 33, 48, … (расстояние до дна умножается на $1-2\cdot1.1=-1.2$, а по модулю это больше 1). Вместо спуска — раскачка, спуск **расходится**. Ровно поэтому шаг нельзя брать сколько хочется.

**Подумайте.** А что будет при $\eta=0.1,\ 0.5,\ 0.9,\ 1.0$? Какие сойдутся? Какой быстрее всех? Какой «застрянет» на месте?

In [ ]:
def descend_1d(eta, n_steps=12, x0=-4.0):
    x_t = x0
    path = [x_t]
    for _ in range(n_steps):
        x_t = x_t - eta * grad_f(x_t)   # шаг вниз против градиента
        path.append(x_t)
    return path

### ✍ Попробуйте сами
Впишите в список `etas_try` значения 0.1, 0.5, 0.9, 1.0 (по одному или сразу все) и запустите. Сверьте с тем, что предсказали: какие шаги сходятся, какой быстрее всех, какой качается на месте? Потом добавьте 0.99 и 1.01 и найдите, где спуск перестаёт сходиться.

In [ ]:
etas_try = [0.1]          # <- добавляйте значения: 0.5, 0.9, 1.0, 0.99, 1.01
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for eta in etas_try:
    ax.plot(descend_1d(eta, n_steps=15), 'o-', label=f'η = {eta}')
ax.axhline(0, color='k', ls='--', lw=1)
ax.set_ylim(-14, 14)
ax.set_xlabel('номер шага'); ax.set_ylabel('x'); ax.legend(fontsize=8)
ax.set_title('Дно в x = 0 (пунктир)');

**Что должно было получиться:**
- $\eta=0.1$ — сходится, но **медленно** (шаг слишком мелкий);
- $\eta=0.5$ — **сразу в цель** (за один шаг попали в 0);
- $\eta=0.9$ — сходится, но **прыгает** через дно туда-сюда;
- $\eta=1.0$ — **качается** между $-4$ и $4$ и никогда не сойдётся (это граница);
- $\eta>1$ — **расходится**, как мы видели при $\eta=1.1$.

**Почему так.** Один шаг умножает расстояние до дна на $(1-2\eta)$. Сходится, пока $|1-2\eta|<1$, то есть $\eta<1$.

**Главное правило:** чем круче парабола (чем больше её «кривизна»), тем меньше шаг допустим. Для $f=a\,x^2$ порог равен $\eta<1/a$.

### Алгоритм градиентного спуска
1. Выбрать стартовую точку $\theta^{0}$ и длину шага $\eta$.
2. Повторять: $\ \theta \leftarrow \theta - \eta\,\nabla Q(\theta)$.
3. Остановиться, когда $\|\theta_{\text{новая}}-\theta_{\text{старая}}\|\le\varepsilon$ (или кончилось заданное число шагов).

Всё занятие построено вокруг одного вопроса: **какое $\eta$ выбрать**.

---
# Часть 2. Вытянутая долина (две переменные)

Теперь функция от двух переменных: по $\theta_1$ парабола пологая, по $\theta_2$ — крутая (в 9 раз):
$$Q(\theta)=(\theta_1-3)^2+9\,(\theta_2-2)^2.$$
Линии уровня — вытянутые эллипсы. Крутость по направлениям задают числа 1 и 9 (это собственные числа $A=\mathrm{diag}(1,9)$; для диагональной матрицы просто её диагональные элементы — «насколько круто вдоль каждой оси»).

**Подумайте.** Порог шага определяется крутым или пологим направлением? Чему он равен?

In [ ]:
A = np.diag([1.0, 9.0])
theta_star = np.array([3.0, 2.0])

def grad_Q(theta):
    return 2 * A @ (theta - theta_star)

In [ ]:
def gradient_descent(theta0, eta, n_iter):
    theta = np.array(theta0, dtype=float)
    hist = [theta.copy()]
    for _ in range(n_iter):
        theta = theta - eta * grad_Q(theta)
        hist.append(theta.copy())
    return np.array(hist)

In [ ]:
eta_max = 1 / np.linalg.eigvalsh(A).max()
print('порог η_max = 1 / (самое большое собственное число) =', round(eta_max, 4))

In [ ]:
theta0 = np.array([-6.0, 5.0])
hist_small = gradient_descent(theta0, 0.8 * eta_max, 40)
hist_big   = gradient_descent(theta0, 1.15 * eta_max, 15)

In [ ]:
g = np.linspace(-9, 9, 300)
T0, T1 = np.meshgrid(g, g)
Qg = (T0 - 3) ** 2 + 9 * (T1 - 2) ** 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, hist, title in zip(axes, [hist_small, hist_big],
                           ['шаг 0.8·η_max: сходится зигзагом', 'шаг 1.15·η_max: расходится']):
    ax.contour(T0, T1, Qg, levels=[2, 10, 30, 60, 120], cmap='viridis')
    ax.plot(hist[:, 0], hist[:, 1], '.-', color='tab:red', ms=4, lw=1)
    ax.plot(*theta_star, 'k*', ms=12)
    ax.set_xlim(-9, 9); ax.set_ylim(-9, 9); ax.set_aspect('equal'); ax.set_title(title, fontsize=10)
plt.tight_layout()

**Что видим:**
- порог определяется **самым крутым** направлением: $\eta_{\max}=1/\lambda_{\max}=1/9$;
- при этом вдоль пологого направления спуск идёт медленно — отсюда **зигзаг**;
- чем сильнее вытянуты эллипсы (чем больше $\lambda_{\max}/\lambda_{\min}$), тем хуже: шаг ограничен крутым направлением, а идти надо по пологому.

Отношение $\lambda_{\max}/\lambda_{\min}$ называется **числом обусловленности** $\mathrm{cond}$.

---
# Часть 3. Проверяем порог шага на регрессии

Возвращаемся к задаче из части 0: **цена квартиры** по **площади**. Функция потерь — сумма квадратов ошибок (это $n\cdot$MSE из части 0):
$$Q(\theta)=\|X\theta-y\|^2,\qquad \nabla Q=2X^\top(X\theta-y).$$
Формулу градиента $2X^\top(X\theta-y)$ вы уже проверяли на прошлом занятии. Из лекции: спуск сходится при $\eta<\eta_{\max}=1/\lambda_{\max}(X^\top X)$. Роль числа «9» из части 2 здесь играет $\lambda_{\max}(X^\top X)$.

**Подумайте.** Возьмём безопасный шаг $0.9\,\eta_{\max}$. Дойдём ли мы до точного решения за 100 шагов? А за 5000?

In [ ]:
def Q_lin(theta, X, y):
    r = X @ theta - y
    return r @ r

In [ ]:
def grad_lin(theta, X, y):
    return 2 * X.T @ (X @ theta - y)

In [ ]:
def gd_curve(eta, n_iter, X, y):
    theta = np.zeros(X.shape[1])
    qs = [Q_lin(theta, X, y)]
    path = [theta.copy()]                                     # траектория θ (нужна для рисунков)
    for _ in range(n_iter):
        theta = theta - eta * grad_lin(theta, X, y)
        qs.append(Q_lin(theta, X, y))
        path.append(theta.copy())
        if not np.isfinite(qs[-1]) or qs[-1] > 1e12 * qs[0]:
            break
    return theta, np.array(qs), np.array(path)

In [ ]:
# x, y, X1 (признак и столбец единиц) мы уже подготовили в части 0

lam_max = np.linalg.eigvalsh(X1.T @ X1).max()
eta_max_raw = 1 / lam_max
theta_star_raw = theta_normal                              # точное решение уже найдено формулой в части 0
print(f'λ_max = {lam_max:.0f},  η_max = {eta_max_raw:.2e}')
print('точное θ* =', theta_star_raw.round(2))

In [ ]:
theta_ok,  q_ok,  _ = gd_curve(0.90 * eta_max_raw, 100, X1, y)

In [ ]:
print(f'после 100 шагов: θ = {theta_ok.round(2)}  (точное θ* = {theta_star_raw.round(2)})')
print(f'расстояние до θ* = {np.linalg.norm(theta_ok - theta_star_raw):.3f}')
theta_5000, _, _ = gd_curve(0.90 * eta_max_raw, 5000, X1, y)
print(f'после 5000 шагов расстояние до θ* = {np.linalg.norm(theta_5000 - theta_star_raw):.3f}   (почти не изменилось)')

**Что видим:**
- за 100 шагов мы **ещё не дошли** до точного решения (расстояние ≈ 0.06, а само решение — порядка 0.1). Даже **5000 шагов почти ничего не меняют**: наш шаг ограничен, а спуск идёт вдоль пологого направления очень медленно.

---
# Часть 4. Стандартизация признаков

В части 3 признак `area` имеет значения ~ 25–150, среднее около 70 (не ноль и совсем другой масштаб, чем у столбца единиц). Из-за этого долина сильно вытянута (большое $\mathrm{cond}$).

**Стандартизация:** $z=(x-\bar x)/\sigma_x$. Тогда среднее равно 0 (центрирование), разброс равен 1 (масштаб). Работают оба эффекта.

**Подумайте.** Как изменится $\mathrm{cond}(X^\top X)$ и сколько шагов теперь понадобится?

In [ ]:
z = (x - x.mean()) / x.std()
Xz = np.column_stack([np.ones_like(z), z])

eig_raw = np.linalg.eigvalsh(X1.T @ X1)
eig_z   = np.linalg.eigvalsh(Xz.T @ Xz)
print(f'без стандартизации: cond = {eig_raw.max()/eig_raw.min():.0f},  η_max = {1/eig_raw.max():.1e}')
print(f'со стандартизацией: cond = {eig_z.max()/eig_z.min():.1f},  η_max = {1/eig_z.max():.1e}')

In [ ]:
theta_star_z = np.linalg.solve(Xz.T @ Xz, Xz.T @ y)
theta_z, q_z, _ = gd_curve(0.9 / eig_z.max(), 100, Xz, y)
print(f'\nсо стандартизацией, 100 шагов: расстояние до θ*_z = {np.linalg.norm(theta_z - theta_star_z):.1e}')
print(f'без стандартизации, 100 шагов: расстояние до θ* = {np.linalg.norm(theta_ok - theta_star_raw):.1e}')
q_star_raw = Q_lin(theta_star_raw, X1, y)                   # значения Q на дне (для линий уровня ниже)
q_star_z   = Q_lin(theta_star_z, Xz, y)

**Что видим:**
- $\mathrm{cond}$ упало с $\approx 5\cdot10^4$ до 1, порог $\eta_{\max}$ вырос примерно в 5400 раз, и **100 шагов дают точность $10^{-9}$**, а без стандартизации те же 100 шагов дают лишь $0.06$;

### Смотрим на рельеф
Линии уровня $Q(w_0,w_1)$ (эллипсы, как на прошлом занятии) и путь спуска из точки $(0,0)$ (первые 100 шагов). Слева — без стандартизации, справа — со стандартизацией.

In [ ]:
def Q_grid(X, w0s, w1s):
    W0, W1 = np.meshgrid(w0s, w1s)
    G, b, c = X.T @ X, X.T @ y, y @ y            # Q = θᵀGθ − 2bᵀθ + yᵀy
    return W0, W1, G[0, 0] * W0**2 + 2 * G[0, 1] * W0 * W1 + G[1, 1] * W1**2 - 2 * b[0] * W0 - 2 * b[1] * W1 + c

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, X, eta, theta_s, q_s, w0r, w1r, title in [
        (axes[0], X1, 0.9 * eta_max_raw, theta_star_raw, q_star_raw, (-0.3, 1.5), (-0.02, 0.22), 'без стандартизации: вытянутая долина'),
        (axes[1], Xz, 0.9 / eig_z.max(), theta_star_z, q_star_z, (-2, 16), (-2, 10), 'со стандартизацией: круги')]:
    W0, W1, Qg = Q_grid(X, np.linspace(*w0r, 300), np.linspace(*w1r, 300))
    ax.contour(W0, W1, Qg, levels=q_s + np.geomspace(20, 3e4, 10), cmap='viridis')
    p = gd_curve(eta, 100, X, y)[2]
    ax.plot(p[:, 0], p[:, 1], '.-', color='tab:red', ms=3, lw=0.8)
    ax.plot(*theta_s, 'k*', ms=12)
    ax.set_xlabel('w0'); ax.set_ylabel('w1'); ax.set_title(title, fontsize=10)
axes[1].set_aspect('equal')
plt.tight_layout()

Справа линии уровня — круги, и путь идёт по прямой через центр. Шаг $0.9\,\eta_{\max}$ близок к пределу, поэтому каждый шаг перелетает центр: расстояние до него за шаг умножается на $1-2\cdot0.9=-0.8$ (меняется сторона, а длина уменьшается на 20%). Слева путь упирается в пологое «дно долины» и почти стоит на месте.

### Сколько шагов нужно до остановки
Вернёмся к критерию остановки из части 1: остановимся, когда $\|\theta_{\text{новая}}-\theta_{\text{старая}}\|\le\varepsilon$. Посмотрим на два значения $\varepsilon$.

In [ ]:
def gd_until(X, eta, eps, max_iter=200000):
    theta = np.zeros(X.shape[1])
    for t in range(1, max_iter + 1):
        theta_new = theta - eta * grad_lin(theta, X, y)
        if np.linalg.norm(theta_new - theta) <= eps:
            return theta_new, t
        theta = theta_new
    return theta, max_iter

In [ ]:
for eps in (1e-4, 1e-6):
    th_raw, steps_raw = gd_until(X1, 0.9 * eta_max_raw, eps)
    th_z, steps_z = gd_until(Xz, 0.9 / eig_z.max(), eps)
    print(f'ε = {eps:g}:  без стандартизации {steps_raw:6d} шагов (расстояние до θ* {np.linalg.norm(th_raw - theta_star_raw):.3f}),'
          f'  со стандартизацией {steps_z:3d} шагов (расстояние {np.linalg.norm(th_z - theta_star_z):.1e})')

**Что видим:**
- при $\varepsilon=10^{-6}$ без стандартизации нужно **больше 20 тысяч шагов**, со стандартизацией — **около 75** (в 300 раз меньше);
- при $\varepsilon=10^{-4}$ без стандартизации спуск **останавливается через 36 шагов**, хотя до решения ещё далеко (расстояние 0.06); со стандартизацией — через 55 шагов, но уже практически в решении (расстояние $4\cdot10^{-5}$). В длинной пологой долине каждый шаг крошечный, и критерий «точка почти не двигается» срабатывает раньше времени. Плохая обусловленность обманывает не только скорость, но и критерий остановки.

**Практическое правило:** перед градиентным спуском всегда стандартизуйте признаки.

---
# Часть 5. ✍ Ваше задание: пишем градиентный спуск сами

Теперь берём **все 4 признака** квартиры, стандартизованные. Готовим данные (ячейка ниже запускается как есть) и смотрим на размеры массивов: с ними чаще всего путаются.

In [ ]:
X_all = flats[features].values
scaler_flats = StandardScaler()
Xs = scaler_flats.fit_transform(X_all)            # стандартизованные признаки
Xb_all = np.column_stack([np.ones(len(y)), Xs])     # + столбец единиц
n, d = Xb_all.shape
print('объектов:', n, ' столбцов (с единицей):', d)

theta_demo = np.zeros(d)
print('X:', Xb_all.shape, ' θ:', theta_demo.shape, ' Xθ − y:', (Xb_all @ theta_demo - y).shape, ' Xᵀ(Xθ − y):', (Xb_all.T @ (Xb_all @ theta_demo - y)).shape)

Градиент $2X^\top(X\theta-y)$ — это вектор той же длины, что и $\theta$ (по одному числу на каждый вес). Из формулы алгоритма (часть 1) нам нужно всего три шага в цикле: посчитать градиент, сдвинуть $\theta$, записать $Q$.

### Задание 5.1. Функция `gd_fit`

Напишите функцию `gd_fit(X, y, eta, n_iter)`, которая:
1. начинает с $\theta=0$ (вектор нулей длины `X.shape[1]`);
2. делает `n_iter` шагов $\ \theta \leftarrow \theta - \eta\cdot 2X^\top(X\theta-y)$;
3. возвращает `theta` и список значений $Q(\theta)=\|X\theta-y\|^2$ **на каждом шаге** (включая начальный).

**Подсказка:** цикл почти такой же, как `gd_curve` в части 3.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ
def gd_fit(X, y, eta, n_iter):
    # ваш код здесь: 5–8 строк
    pass

### Задание 5.2. Применяем ко всем признакам

1. Посчитайте `eta_max_5 = 1 / λ_max(XᵀX)` через `np.linalg.eigvalsh(Xb_all.T @ Xb_all).max()`.
2. Запустите `gd_fit` с `eta = 0.9 * eta_max_5`, `n_iter = 150`.
3. Постройте график `losses` (логарифмическая шкала по оси Y).
4. Сравните `theta_gd` с точным решением `theta_star_all = np.linalg.solve(Xb_all.T @ Xb_all, Xb_all.T @ y)` (это то, что давала ваша `ols_fit` на прошлом занятии).

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ
# eta_max_5 = ...
# theta_gd, losses = gd_fit(...)
# theta_star_all = ...
# график и сравнение

**Проверка.** Запустите ячейку ниже: она проверит функцию и сравнит с точным решением. (Функция должна вернуть длину `losses` = 151, а расстояние до точного решения — совсем крошечное.)

In [ ]:
try:
    eta_max_5 = 1 / np.linalg.eigvalsh(Xb_all.T @ Xb_all).max()
    theta_chk, losses_chk = gd_fit(Xb_all, y, 0.9 * eta_max_5, 150)
    theta_star_all = np.linalg.solve(Xb_all.T @ Xb_all, Xb_all.T @ y)
    dist = np.linalg.norm(theta_chk - theta_star_all)
    ok = (len(losses_chk) == 151) and (dist < 1e-6) and (losses_chk[-1] < losses_chk[0])
    print(f'расстояние до θ*: {dist:.1e}, длина losses: {len(losses_chk)}  ->', '✅ верно' if ok else '❌ что-то не так')
    print(f'MSE: ваш спуск {losses_chk[-1] / n:.4f}, точное решение {np.sum((Xb_all @ theta_star_all - y) ** 2) / n:.4f}')
except Exception as e:
    print('❌ функция ещё не готова:', type(e).__name__, e)

**Сначала руками, потом библиотекой.** Сравним наш спуск с готовой `LinearRegression` из `sklearn`: она решает ту же задачу на тех же стандартизованных признаках (столбец единиц sklearn добавляет сам).

In [ ]:
try:
    lr = LinearRegression().fit(Xs, y)
    theta_sk = np.concatenate([[lr.intercept_], lr.coef_])
    print('sklearn:  ', theta_sk.round(4))
    print('наш спуск:', theta_chk.round(4))
    print(f'расстояние: {np.linalg.norm(theta_sk - theta_chk):.1e}')
except NameError:
    print('сначала добейтесь ✅ в ячейке выше')

**Ожидаемо:** расстояние совсем крошечное, порядка $10^{-11}$ (почти совпадает с точным решением), MSE как у точного решения, кривая на логарифмической шкале — почти прямая линия (сходимость геометрическая).

Почему 150 шагов, а не около 75, как в части 4? Здесь признаков несколько, и два из них скоррелированы (`area` и `rooms`: чем больше квартира, тем больше комнат), поэтому $\mathrm{cond}(X^\top X)\approx 12$, а не 1.

### Получили ли мы «истинные» веса?
Мы решили задачу на **стандартизованных** признаках, поэтому веса «в стандартных отклонениях». Чтобы сравнить с истиной, вернём их в исходные единицы: делим каждый вес на стандартное отклонение своего признака.

In [ ]:
theta_exact = np.linalg.solve(Xb_all.T @ Xb_all, Xb_all.T @ y)          # точное решение
w_orig = theta_exact[1:] / X_all.std(axis=0)                            # веса в исходных единицах
print(pd.DataFrame({'признак': features,
                    'истинный вес': [TRUE_WEIGHTS[f] for f in features],
                    'нашли': w_orig.round(3)}).to_string(index=False))

Веса получились близкими к истинным (все знаки совпали); небольшое расхождение — из-за шума в данных и всего 300 квартир.

### Прогноз цены
Ради этого всё и затевалось: модель готова предсказывать цену. Посчитаем прогноз для новой квартиры (60 м², 2 комнаты, 5-й этаж, 1 км до метро) и посмотрим, насколько прогнозы близки к настоящим ценам.

In [ ]:
new_flat = pd.DataFrame({'area': [60.0], 'rooms': [2.0], 'floor': [5.0], 'metro_km': [1.0]})
z_new = scaler_flats.transform(new_flat[features].values)            # та же стандартизация, что при обучении
price_new = theta_exact[0] + z_new[0] @ theta_exact[1:]
price_true = 1.0 + sum(TRUE_WEIGHTS[f] * new_flat[f][0] for f in features)
print(f'прогноз: {price_new:.2f} млн ₽;  по «истинной» формуле без шума: {price_true:.2f} млн ₽')

pred = Xb_all @ theta_exact
rmse = np.sqrt(np.mean((pred - y) ** 2))
plt.figure(figsize=(4.8, 4.8))
plt.scatter(y, pred, s=8, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=1)
plt.xlabel('настоящая цена, млн ₽'); plt.ylabel('прогноз, млн ₽')
plt.title(f'прогноз против факта: RMSE = {rmse:.2f} млн ₽');

**Что видим:** прогноз для новой квартиры — 7.24 млн ₽, по «истинной» формуле 7.30. Точки лежат вдоль диагонали, типичная ошибка прогноза (RMSE) ≈ 1 млн ₽ — это ровно шум, который мы заложили в данные ($\sigma=1$). Точнее на этих данных не может никакая модель: остальное — случайность.

---
# Часть 6. SGD: когда он выигрывает и как его применять

До сих пор на каждом шаге мы считали градиент по **всем** объектам. Это дорого, если объектов миллион. Идея: считать градиент по **случайной части** объектов:

| Способ | Объектов в шаге | Особенность |
|---|---|---|
| Полный градиент | все | гладко, но шаг дорогой |
| Мини-батч | 32, 256, … | компромисс (свой мини-батч вы напишете в домашней работе) |
| SGD | 1 | дёшево, но шумно |

**Когда SGD выигрывает.** Нужны три условия: (1) объектов очень много ($n\gg d$); (2) объекты «повторяют» друг друга, поэтому небольшая случайная часть выборки даёт почти тот же градиент, что и вся; (3) признаки стандартизованы, иначе ползёт и SGD. Поэтому сгенерируем **200 000 квартир**.

Сравнивать будем не по числу шагов (шаги стоят по-разному), а по **числу обработанных объектов**: именно за столько вычислений градиента мы платим. Потери теперь — среднее (MSE), чтобы сравнивать способы на одной шкале.

**Подумайте.** Сколько объектов должен обработать полный градиент, чтобы дойти до уровня МНК (с запасом 2%), и сколько — SGD? Что будет с SGD, если шаг не уменьшать?

In [ ]:
flats_big = make_flats(200000, seed=1)                                   # много квартир
y_big = flats_big['price'].values
Xs_big = StandardScaler().fit_transform(flats_big[features].values)      # стандартизованные признаки
Xb_big = np.column_stack([np.ones(len(y_big)), Xs_big])                  # + столбец единиц (для формулы и полного градиента)
n_big = len(y_big)

Точное решение формулой и полный градиент для сравнения — нашей `gd_curve` из части 3 с лучшим безопасным шагом $0.9/\lambda_{\max}(X^\top X)$. Один шаг полного градиента обрабатывает все $n$ объектов.

In [ ]:
theta_big = np.linalg.solve(Xb_big.T @ Xb_big, Xb_big.T @ y_big)       # точное решение
mse_ols = np.mean((Xb_big @ theta_big - y_big) ** 2)

eta_full = 0.9 / np.linalg.eigvalsh(Xb_big.T @ Xb_big).max()
_, q_full, _ = gd_curve(eta_full, 30, Xb_big, y_big)                     # 30 шагов = 30 проходов по данным
curves = {'полный градиент': (np.arange(len(q_full)) * n_big, q_full / n_big)}   # (обработано объектов, MSE)

### SGD из `sklearn`: `SGDRegressor`
Это линейная регрессия, которую обучают стохастическим градиентным спуском (по одному объекту за шаг). Главные параметры:
- `penalty=None` — без штрафа. По умолчанию включён небольшой L2-штраф (про штрафы — в части 7), нам он сейчас не нужен;
- `learning_rate` и `eta0` — шаг: `'constant'` — постоянный $\eta_0$; по умолчанию `'invscaling'` — затухающий, $\eta_t=\eta_0/t^{0.25}$;
- свободный член модель добавляет сама, поэтому подаём `Xs_big` **без** столбца единиц.

Обычно пишут просто `model.fit(X, y)`. Нам же нужна кривая обучения, поэтому подаём объекты порциями через `partial_fit` (один проход по переданным объектам) и после каждой порции считаем MSE на всей выборке.

In [ ]:
def sgd_curve(model, budget, seed=1):
    # подаём объекты порциями (по «геометрической» сетке) и после каждой порции считаем MSE на всей выборке
    idx = np.random.default_rng(seed).permutation(n_big)
    marks = np.unique(np.geomspace(10, budget, 50).astype(int))
    seen, losses, done = [0], [np.mean(y_big ** 2)], 0          # старт: все веса нулевые
    for mark in marks:
        while done < mark:
            start = done % n_big
            part = idx[start:start + min(mark - done, n_big - start)]
            model.partial_fit(Xs_big[part], y_big[part])
            done += len(part)
        seen.append(done); losses.append(np.mean((model.predict(Xs_big) - y_big) ** 2))
    return np.array(seen), np.array(losses)

Две модели: с постоянным шагом и с шагом по умолчанию (затухающим). Каждой даём обработать 600 000 объектов (3 прохода по данным):

In [ ]:
sgd_const = SGDRegressor(penalty=None, learning_rate='constant', eta0=0.01, random_state=0)
sgd_decay = SGDRegressor(penalty=None, random_state=0)             # шаг по умолчанию: затухающий
curves['SGD, постоянный шаг'] = sgd_curve(sgd_const, 3 * n_big)
curves['SGD, затухающий шаг'] = sgd_curve(sgd_decay, 3 * n_big)

Кривые: по горизонтали — число обработанных объектов (логарифмическая шкала):

In [ ]:
plt.figure(figsize=(7.5, 4.5))
for label, (seen, losses) in curves.items():
    plt.plot(seen[1:], losses[1:], label=f'{label}: MSE = {losses[-1]:.4f}')
plt.axhline(mse_ols, color='k', ls='--', lw=1, label=f'МНК: {mse_ols:.4f}')
plt.xscale('log'); plt.ylim(0.95, 2)
plt.xlabel('обработано объектов'); plt.ylabel('MSE'); plt.legend(fontsize=8);

Сколько объектов понадобилось, чтобы дойти до уровня МНК с запасом 2% (MSE ≤ 1.02 · MSE МНК), и совпали ли веса:

In [ ]:
rows = []
for label, (seen, losses) in curves.items():
    hit = np.where(losses <= 1.02 * mse_ols)[0]
    rows.append({'способ': label, 'объектов до цели': int(seen[hit[0]]), 'MSE в конце': round(losses[-1], 4)})
need = pd.DataFrame(rows)
need['выигрыш, раз'] = (need['объектов до цели'].iloc[0] / need['объектов до цели']).round(0)
print(need.to_string(index=False))

weights = pd.DataFrame({'МНК': theta_big,
                        'SGD, постоянный шаг': np.r_[sgd_const.intercept_, sgd_const.coef_],
                        'SGD, затухающий шаг': np.r_[sgd_decay.intercept_, sgd_decay.coef_]},
                       index=['сдвиг'] + features)
print('\n', weights.round(3).to_string())

**Что видим:**
- полному градиенту до уровня МНК даже с лучшим шагом нужно **≈2.8 млн обработанных объектов** (14 проходов по данным), SGD — **2–7 тысяч**: выигрыш в сотни и тысячи раз;
- причина в повторяемости данных: объектов 200 тысяч, а модель — всего пять чисел, поэтому уже горстка случайных объектов показывает верное направление. Чем больше $n$, тем больше выигрыш;
- **постоянный шаг** спускается быстрее всех, но потом дрожит и остаётся выше МНК (MSE 1.008 против 0.999), веса неточные (у `rooms` 0.24 вместо 0.30): шум одного объекта не даёт сесть на дно;
- **затухающий шаг** (так `SGDRegressor` работает по умолчанию) стартует медленнее, зато шум гаснет: MSE и веса почти как у МНК.

**Когда выигрыша не будет:** если объектов мало (сотни–тысячи), экономия невелика, а шум её съедает; без стандартизации медленно ползут оба способа.

**Как применять:** стандартизовать признаки и вызвать `SGDRegressor(penalty=None).fit(X, y)` (или со штрафом — часть 7), шаг оставить затухающим.

---
# Часть 7. Регуляризация: Ridge и LASSO

### Зачем это нужно: переобучение на маленьком примере
Истинная зависимость — кубическая парабола $f(x)=0.4+2x+0.8x^2+0.5x^3$ (зелёная линия). Нам дали всего **10 зашумлённых точек**, а модель мы берём слишком гибкую: многочлен степени 11 (11 признаков $x, x^2, \dots, x^{11}$, стандартизованных, как учили в части 4). Проверять будем на 200 новых точках без шума.

**Подумайте.** Что сделает обычная линейная регрессия с 11 признаками на 10 точках?

In [ ]:
def f_true(x):
    return 0.4 + 2 * x + 0.8 * x ** 2 + 0.5 * x ** 3

rng_toy = np.random.default_rng(18)
x_train = rng_toy.uniform(-5, 5, 10)
y_train = f_true(x_train) + 10 * rng_toy.standard_normal(10)      # 10 точек с шумом
x_test = np.sort(np.random.default_rng(8).uniform(-5, 5, 200))
y_test = f_true(x_test)                                          # «чистая истина» для проверки

def plot_toy():
    # истинная зависимость (зелёная) и обучающие точки (чёрные)
    plt.figure(figsize=(8, 4.5))
    plt.plot(x_test, y_test, 'g', lw=2, label='истинная f(x)')
    plt.scatter(x_train, y_train, c='k', zorder=3, label='10 точек для обучения')
    plt.xlabel('x'); plt.ylabel('y')

In [ ]:
poly = PolynomialFeatures(11, include_bias=False)
scaler = StandardScaler()
Xp_train = scaler.fit_transform(poly.fit_transform(x_train.reshape(-1, 1)))
Xp_test = scaler.transform(poly.transform(x_test.reshape(-1, 1)))         # те же среднее и разброс, что у обучающих

In [ ]:
def fit_linear(X, y_):
    m = LinearRegression().fit(X, y_)
    return m.coef_, m.intercept_

def fit_ridge(X, y_, alpha):
    m = Ridge(alpha=alpha).fit(X, y_)
    return m.coef_, m.intercept_

In [ ]:
def fit_lasso(X, y_, alpha):
    m = Lasso(alpha=alpha, max_iter=200000, random_state=0).fit(X, y_)
    return m.coef_, m.intercept_

def mse_of(model, X, y_):
    w, b = model
    return np.mean((X @ w + b - y_) ** 2)

In [ ]:
lin = fit_linear(Xp_train, y_train)
plot_toy()
plt.plot(x_test, Xp_test @ lin[0] + lin[1], 'r', label='обычная регрессия')
plt.ylim(-60, 100); plt.legend(fontsize=8)

In [ ]:
print(f'ошибка на обучающих точках: {mse_of(lin, Xp_train, y_train):.1e}')
print(f'ошибка на новых точках:     {mse_of(lin, Xp_test, y_test):.1e}')
print(f'наибольший по модулю вес:   {np.abs(lin[0]).max():.0f}')

**Что видим:** красная кривая проходит **ровно через все 10 точек** (ошибка на обучении почти нуль), но между ними «пляшет», и на новых точках ошибка огромна (порядка $10^8$). Веса раздуты до десятков тысяч. Это **переобучение**: модель запомнила шум.

**Идея регуляризации:** штрафовать за большие веса. Обучаем не просто $\|Xw-y\|^2$, а
$$\|Xw-y\|^2+\alpha\,\|w\|_2^2\ \ \text{(Ridge)},\qquad \|Xw-y\|^2+\alpha\,\|w\|_1\ \ \text{(LASSO)}.$$
Число $\alpha$ (сила штрафа) выбираем сами. У `Ridge` в `sklearn` целевая функция ровно такая; у `Lasso` целевая функция нормирована иначе: $\frac1{2n}\|Xw-y\|^2+\alpha\|w\|_1$, то есть $\alpha$ отличается от формулы выше множителем $2n$ (качественная картина та же). Если штраф считать в единицах MSE, то $\text{MSE}+\lambda\|w\|_2^2$ соответствует `Ridge(alpha=n·λ)`, а $\text{MSE}+\lambda\|w\|_1$ — `Lasso(alpha=λ/2)`; так это будет записано в домашней работе. Считать будем готовыми моделями `Ridge` и `Lasso` из `sklearn`; у Ridge есть и формула $w=(X^\top X+\alpha E)^{-1}X^\top y$, у LASSO формулы нет.

**Подумайте.** Что будет с кривой при очень маленьком $\alpha$? При очень большом? Где, по-вашему, лучшее $\alpha$?

In [ ]:
alphas_ridge = [0.001, 0.1, 1, 10, 100]
ridge_models = {a: fit_ridge(Xp_train, y_train, a) for a in alphas_ridge}

In [ ]:
plot_toy()
for a, (w, b) in ridge_models.items():
    plt.plot(x_test, Xp_test @ w + b, '--', label=f'Ridge, alpha = {a}')
plt.ylim(-60, 100); plt.legend(fontsize=8);

In [ ]:
table = pd.DataFrame([{'alpha': a,
                       'ошибка на обучении': mse_of(m, Xp_train, y_train),
                       'ошибка на новых точках': mse_of(m, Xp_test, y_test),
                       'наибольший вес': np.abs(m[0]).max()} for a, m in ridge_models.items()])
table['alpha'] = table['alpha'].astype(str)
print(table.round(1).to_string(index=False))

**Что видим:**
- даже самый малый штраф ($\alpha=0.001$) сразу убрал «пляску»: кривая стала гладкой, хотя она ещё гуляет вокруг истины (ошибка на новых точках 65, а не $10^8$);
- при большом $\alpha$ кривая становится всё более плоской и уходит от истины: модель **недообучилась** (штраф забил данные; при $\alpha=100$ ошибка 309);
- лучшая по ошибке на **новых** точках — где-то посередине ($\alpha\approx1$: ошибка около 40 вместо $10^8$). Заметьте: ошибка на обучении с ростом $\alpha$ только **растёт** (12 → 16 → 30 → 71 → 394). Регуляризация меняет точность на обучении на устойчивость;
- ошибка на новых точках не монотонна по $\alpha$: точек мало, поэтому картина шумная. Так всегда бывает на маленьких выборках.

Посмотрим на веса: как они меняются с ростом $\alpha$ для Ridge и LASSO.

In [ ]:
alphas_path = np.logspace(-2, 2, 40)
path_ridge = np.array([fit_ridge(Xp_train, y_train, a)[0] for a in alphas_path])
path_lasso = np.array([fit_lasso(Xp_train, y_train, a)[0] for a in alphas_path])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
axes[0].plot(alphas_path, path_ridge, lw=2); axes[0].set_title('Ridge: веса плавно уменьшаются')
axes[1].plot(alphas_path, path_lasso, lw=2); axes[1].set_title('LASSO: веса обнуляются')
for ax in axes:
    ax.set_xscale('log'); ax.axhline(0, color='gray', lw=0.5); ax.set_xlabel('alpha')
axes[0].set_ylabel('вес')
plt.tight_layout()

**Что видим:** у Ridge веса уменьшаются **плавно**, но ни один не становится ровно нулём. У LASSO веса **обнуляются** (обычно один за другим, но не всегда по порядку), то есть LASSO ещё и **отбирает признаки**.

---
# Часть 8. Найди ошибку

Коллега взял шаг $\eta=0.9\,\eta_{\max}$ из части 5 (он отлично работал на **стандартизованных** признаках) и применил его к **исходным** 4 признакам (не стандартизованным). Код не падает, но происходит что-то странное.

**Подумайте.** Что будет с $\theta$? Почему?

In [ ]:
Xb_raw = np.column_stack([np.ones(n), X_all])      # без стандартизации!
eta_z = 0.9 / np.linalg.eigvalsh(Xb_all.T @ Xb_all).max()          # шаг из части 5
lam_raw = np.linalg.eigvalsh(Xb_raw.T @ Xb_raw).max()

print(f'шаг коллеги: η = {eta_z:.1e}')
print(f'а порог для этих данных: η_max = {1/lam_raw:.1e}  -> шаг больше порога в {eta_z*lam_raw:.1e} раз')

In [ ]:
theta_bug = np.zeros(Xb_raw.shape[1])
with np.errstate(over='ignore', invalid='ignore'):
    for t in range(60):
        theta_bug = theta_bug - eta_z * grad_lin(theta_bug, Xb_raw, y)
        if t in (0, 1, 2, 5, 10, 20, 40, 59):
            print(f'итерация {t:2d}: max|θ| = {np.max(np.abs(theta_bug)):.2e}')

**Ответ.** У нестандартизованных признаков масштабы огромные (`area` — десятки и сотни, `floor` — единицы и десятки, `metro_km` — единицы), поэтому $\lambda_{\max}(X^\top X)$ **намного больше**, а порог $\eta_{\max}=1/\lambda_{\max}$ — **намного меньше**, чем шаг для стандартизованных данных. Шаг превысил порог — спуск расходится, $\theta$ растёт с каждой итерацией.

**Как исправить:** либо стандартизовать признаки, либо пересчитать $\eta_{\max}$ для *этих* данных.

**Урок:** шаг нельзя переносить с одних данных на другие. Порог всегда считается заново.

---
## Итоги занятия

1. Спуск с постоянным шагом сходится при $\eta<1/\lambda_{\max}$ и расходится при большем шаге.
2. Если признаки в разных масштабах, эллипсы вытянуты ($\mathrm{cond}$ большое), спуск идёт зигзагом и медленно. **Стандартизация** делает эллипсы кругами.
3. **SGD** (в `sklearn` — `SGDRegressor`): один шаг дешёвый, кривая шумнее. На больших однородных данных он доходит до уровня МНК, обработав в сотни раз меньше объектов, чем полный градиент. Постоянный шаг оставляет «пол» из шума, затухающий шаг его убирает.
4. **Регуляризация** борется с переобучением: штрафуем большие веса. **Ridge** уменьшает коэффициенты плавно, **LASSO** зануляет (отбирает признаки). Силу штрафа выбирают по ошибке на отложенных данных.
5. Спуск останавливают по числу шагов или когда точка перестала двигаться ($\|\Delta\theta\|\le\varepsilon$). При малом $\varepsilon$ со стандартизацией до остановки нужно в сотни раз меньше шагов, а при большом $\varepsilon$ без стандартизации критерий срабатывает раньше, чем дойдём до решения.

### Когда что брать
Всё это — одна и та же линейная модель; разница в том, **как** ищутся веса.

| Что | Как ищет веса | Когда брать |
|---|---|---|
| `LinearRegression` | формулой (через SVD), точно, за один вызов | **по умолчанию**, если данные помещаются в память (до сотен тысяч объектов и сотен признаков) |
| `SGDRegressor` | стохастическим градиентным спуском, приближённо | данных очень много (миллионы строк), они не помещаются в память или приходят потоком (`partial_fit`); признаки обязательно стандартизовать |
| `Ridge`, `Lasso`, `ElasticNet` | формулой или итерационно, **со штрафом** | признаков много или они почти дублируют друг друга, данных мало, есть риск переобучения (часть 7) |
| свой градиентный спуск | руками | чтобы понять, что происходит внутри `fit()`; на практике не нужен |

`SGDRegressor(penalty=None)` сходится к тем же весам, что и `LinearRegression`, а `SGDRegressor(penalty='l2')` — это Ridge, посчитанный через SGD. На наших 300 квартирах правильный выбор — `LinearRegression`; 200 000 квартир в части 6 мы взяли нарочно, чтобы увидеть, где SGD выигрывает.

### Шпаргалка «что-то пошло не так»
| Симптом | Причина | Что делать |
|---|---|---|
| $\theta$ или $Q$ растут, потом `inf`/`nan` | шаг больше порога $1/\lambda_{\max}$ | уменьшить $\eta$ |
| $Q$ убывает, но очень медленно | долина вытянута (большое $\mathrm{cond}$) | стандартизовать признаки |
| шаг «работал» на других данных | порог зависит от данных | пересчитать $\eta_{\max}$ заново |
| кривая шумит, не садится на дно | SGD: шум мини-батча | уменьшить шаг (или сделать его затухающим), увеличить batch |

## Домашняя работа 3

Условие — в отдельном ноутбуке `дз3_заготовка`: те же приёмы на личных данных (доставка еды).